In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import time

SEED = 1234
VAL_SPLIT = 0.1 # 10% of dataset used to validate
rng = np.random.default_rng(SEED) # how it should be called in every block requiring it to avoid consistency issues

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu")
print(device) # cuda for support GPUs, mps for apple silicon, cpu otherwise

In [ ]:
class Vectorise:
  """Converts PIL image from FashionMNIST -> flat float32 vector"""
  def __init__(self, mean=None, std=None, rescale=True):
    self.mean, self.std, self.rescale = mean, std, rescale

  def __call__(self, img):
    x = np.array(img, dtype=np.float32).reshape(-1) # 784 values in [0, 255]
    if self.rescale:
      x /= 255. # sends to [0, 1]
    if self.mean is not None and self.std is not None:
      x = (x - self.mean) / self.std
    return x
    # its a class and not a lambda since lambdas will break on windows + macos

# fixed split + normalization statistics (training split only), computed once
train_raw = datasets.FashionMNIST(root='./data', train=True, download=True)
perm = np.random.default_rng(SEED).permutation(len(train_raw))
n_val = int(VAL_SPLIT * len(train_raw))
val_idx, train_idx = perm[:n_val], perm[n_val:]

train_pixels = train_raw.data.numpy()[train_idx].astype(np.float32) / 255.0  # on the [0,1] scale
MEAN, STD = float(train_pixels.mean()), float(train_pixels.std())


def to_arrays(dataset, batch_size=1024):
  """Converts a dataset of (vector, label) pairs into numpy arrays X (N, 784), y (N,)"""
  loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
  Xs, ys = [], []

  for xb, yb in loader:
    Xs.append(xb.numpy())
    ys.append(yb.numpy())

  return np.concatenate(Xs).astype(np.float32), np.concatenate(ys).astype(np.int64)


# -- augmentations
def build_augmentations():
  """
  Returns a list of augmentation transforms to apply to training images
  """
  return [transforms.RandomHorizontalFlip(), transforms.ColorJitter(brightness=0.5, contrast=0.5)]

# call to obtain typical numpy appropriate data!
def load_data(normalize=True):
  """
  Returns X_train, y_train, X_val, y_val, X_test, y_test as numpy arrays
  Inputs:
  normalize:  a boolean flag to normalize the loaded dataset, true by default

  Output:
  X_train, y_train, X_val, y_val, T_test, y_test:   numpy arrays converted from vectorised (vector, label) pairs into (N, 784), y(N,)
  """
  tf = Vectorise(MEAN, STD) if normalize else Vectorise(rescale=False) # unnormalized = raw 0-255 (needed for Q4)

  train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=tf)
  test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=tf) # again, dont augment the test data (obviously!)

  return (*to_arrays(Subset(train_data, train_idx)),
          *to_arrays(Subset(train_data, val_idx)),
          *to_arrays(test_data))


X_train, y_train, X_val, y_val, X_test, y_test = load_data(normalize=True)

# for augmentation
# create a dataset that applies fresh augmentations whenever an image is accessed
aug_transformer = transforms.Compose(build_augmentations() + [Vectorise(MEAN, STD)])
aug_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=aug_transformer)
aug_train = Subset(aug_data, train_idx)


# checks
print(X_train.shape, X_val.shape, X_test.shape)  # (54000, 784) (6000, 784) (10000, 784)
print(X_train.mean(), X_train.std())             # ~0 and ~1 when normalized
print(np.bincount(y_train))                      # ~5400 per class

# T2: Multilayer Perceptron

$$
\hat{y} = \sigma \left ( W \sigma \left ( V x \right ) \right)
$$

with $M$ hidden units and $D$ input features:

$$
w \in \mathbb{R}^{M},  
V \in \mathbb{R}^{M \times D}.
$$

## MLP Class Implementation

Backpropagation + forward passes

Regularization (detailed later)

In [ ]:
# utilities for MLP
# __________________

logistic = lambda z: 1./ (1 + np.exp(-z))

def softmax(z): # 2-dimensional for batches
  z = z - z.max(axis=1, keepdims=True) # numerically stable version subtracting row before exponentiating
  e = np.exp(z)
  return e / e.sum(axis=1, keepdims=True)

def generate_a_func(name, l=0.01):
  """
  Gives a callable elementwise function to apply on numpy arrays
  Inputs:
  name:   'logistic' | 'relu' | 'leaky_relu' | 'tanh'
  l:      hyperparameter to tune for leaky relu, shrinking factor for z < 0
  Output:
  (f, f_prime):   elementwise function and its derivative to apply on np.array
  """
  if name == 'logistic':
      return (lambda z: logistic(z),                    # 1 / 1+e^-z
              lambda z: np.multiply(logistic(z), 1 - logistic(z))) # df/dx = f(x)(1-f(x))
  elif name == 'relu':
      return (lambda z: np.maximum(0, z),            # send to 0
              lambda z: (z > 0).astype(z.dtype))     # { 1 if z > 0; 0 otherwise }
  elif name == 'leaky_relu':
      return (lambda z: np.where(z > 0, z, l * z), # scale by a (hyperparameter to tune!)
              lambda z: np.where(z > 0, 1.0, l))   # { 1 if z > 0; a otherwise }
  elif name == 'tanh':
      return (np.tanh,                               # s shaped
              lambda z: 1.0 - np.tanh(z) ** 2)       # sech^2 = 1-tanh
  else:
    raise ValueError("Pick one of 'logistic' / 'relu' / 'leaky_relu' / 'tanh'")

def evaluate_acc(y, y_hat):
  """
  Evaluates model accuracy to assignment spec
  """
  return np.mean(y == y_hat)

In [ ]:
# MLP
# ___

def cross_entropy(y, p, eps=1e-12):
  """
  Computes cross-entropy loss

  Inputs:
  y:    integer labels, shape (N,)
  p:    softmax probabilities, shape (N, C)
  eps:  small noise
  """
  N = y.shape[0]
  p_true = p[np.arange(N), y]
  # picks the true class out of each. avoids one-hot encoding
  # arange evenly spaces values so we can pick them out as classes
  return -np.mean(np.log(np.clip(p_true, eps, 1.0))) # keeps within bounds


class MLP:

  def __init__(self, hidden=(256, 256), a_func='relu', leaky_factor=0.0, reg=None, lam=0.0, dropout=0.0, seed=SEED):
    """
    Initialize a K-layer neural network multilayer perception

    x -> [W1,b1] -> σ -> ... -> [W_K,b_K] -> σ -> [W_out,b_out] -> softmax

    Inputs:
    hidden:       hidden units per layer, ex. (256, 256) meaning 2 hidden layers of 256 units each.
                  an empty set () indicates 0 hidden layers, reducing MLP to softmax regression
    activation:   activation function of each layer and hidden unit respectively
                  'relu' | 'logistic' | 'leaky_relu' | 'tanh'
    leaky_factor: factor to scale leaky relu by, when selected. 0 by default (reduces to relu)
    reg:          regularization to apply during weight updates
                  'L1' | 'L2' | None
    lam:          regularization strength, lambda. Higher indicates greater regularization strength
    dropout:      0 <= dropout < 1, rate of hidden unit dropout to aid in generalization. 0 reduces to no dropout.

    Call _init_weights(D, C) to initialize the weights and biases before fitting
    """
    self.hidden = list(hidden)
    self.nb_layers = len(hidden) + 1 # +1 for output layer
    self.activation = a_func
    self.a_func, self.da_func = generate_a_func(a_func, l=leaky_factor) # callable elementwise functions
    self.reg = reg
    self.lam = lam
    self.rng = np.random.default_rng(seed) # call again
    self.dropout = np.clip(dropout, 0.0, 0.999)
    self.drop_rng = np.random.default_rng(seed+1) # different from rng
    self.weights = []
    self.biases = []

    # weight initialization: Kaiming/He for ReLU activation, Xavier-Glorot for logistic or tanh
    # helps eliminate gradient vanishing since loss is non-convex, and by avoiding the symmetry problem
    # we'll leave them empty for now since the shape of the dataset isnt known until runtime (since we're augmenting, it changes shape)
    # see init_weights(D, C) below and call it!

  def init_weights(self, D, C):
    """
    Initialize weights of an MLP to be non-zero to avoid symmetry and gradient vanishing.
    Optimal initialization is dependent on the activation function of hidden layers

    Inputs:
    D:    Number of features in the input
    C:    Number of classes in the output

    """
    self.weights, self.biases = [], [] # start from nothing in case we're reinitializing
    layer_sizes = [D] + self.hidden + [C] # layer sizes from input to output, like [784, 256, 256, 10] in default case

    # weights take inputs to next layer
    layer_dims = list(zip(layer_sizes[:-1], layer_sizes[1:])) # [(784, 256), (256, 256), (256, 10)] in default case
    nb_layers = len(layer_dims)

    for layer in range(nb_layers):
      D_in, D_out = layer_dims[layer]
      is_last = layer == nb_layers - 1 # last layer is output

      # check which activation layer used
      if self.activation in ('relu', 'leaky_relu') or is_last: # putting last layer through Kaiming compensates for relu sending many logits to zero
        variance = 2 / D_in # Kaiming/He
      elif self.activation in ('logistic', 'tanh'):
        variance = 2 / (D_in + D_out) # Xavier-Glorot
      else:
        raise ValueError("Something went wrong in initialization!") # technically avoidable because it's needed for init

      # use dynamic variances for random initialization of weights
      std = np.sqrt(variance)
      W = self.rng.normal(loc = 0.0, scale = std, size = (D_in, D_out))
      b = np.zeros(D_out) # ok to leave biases at zero

      self.weights.append(W)
      self.biases.append(b)


  def forward(self, x, training=False):
    """Return softmax logits (N, C) and list (a_prev, z) for backprop"""
    # loop over layers. for each hidden layer, store layers input 'a_prev' and preactivation 'z'
    # then a = self.a_func(z)
    # last layer has no activation. output logits as softmax
    # keep running precomputed values for backprop: list(a_prev, z) per layer
    a = x
    a_prevs = []
    for i in range(self.nb_layers):
      a_prev = a
      z = a_prev @ self.weights[i] + self.biases[i] # @ is dot product
      # a_prevs.append((a_prev, z))

      dropout_mask = None


      if i == self.nb_layers - 1: # last layer is output and is just identity
        a = z
      else:
        a = self.a_func(z) # activate z
        if self.dropout > 0:
          if training: # dropout, only during training, not during predictions
            dropout_mask = (self.drop_rng.random(a.shape) >= self.dropout).astype(a.dtype) # if it exceeds dropout rate, drop
            a = a * dropout_mask # mask array with dropouts per layer
          else:
            a = a * (1-self.dropout) # scale by the non-drop probability to make up for lost weight

      a_prevs.append((a_prev, z, dropout_mask)) # dropout_mask = None when dropout=0.0

    return a, a_prevs # last ones are logits


  def backprop(self, p, y, a_prevs):
    """
    Applies the chain rule from loss back to every W and b, reusing previous computations at each layer

    Inputs:
    p:        softmax probabilities (N,C)
    y:        integer labels (N,)
    a_prevs:  (a_prev, z) for each layer

    Outputs:
    (dW, db): gradients, one array per layer in forward layer (since for loop works backwards)
    """
    # gradients
    dW = [None] * self.nb_layers
    db = [None] * self.nb_layers

    # we do output dZ first
    N = y.shape[0]
    dZ = p.copy()
    dZ[np.arange(N), y] -= 1 # removes the one-hot matrix to obtain p-Y where Y is one-hot, i.e p-1 when true
    dZ = dZ / N # normalize as mean average

    for i in range(self.nb_layers -1, -1, -1): # backwards
      (a_prev, z, dropout_mask) = a_prevs[i] # dropout_mask = None if dropout=0.0

      # compute gradients
      dW[i] = a_prev.T @ dZ
      db[i] = dZ.sum(axis=0) # just the firsts

      # regularization penalty on weights
      if self.reg == 'L2':    # sends closer to 0
        dW[i] += (self.lam * self.weights[i])
      elif self.reg == 'L1':  # sends exactly to 0
        dW[i] += (self.lam * np.sign(self.weights[i]))

      if i > 0: # for all but output
        z_prev = a_prevs[i-1][1] # (a_prev, ->z)
        dropout_prev = a_prevs[i-1][2]
        dZ = dZ @ self.weights[i].T
        if dropout_prev is not None:
          dZ = dZ * dropout_prev # drop units

        dZ = dZ * self.da_func(z_prev) # apply derivative activation function chain rule

    return dW, db

  def fit(self, X, y, C, optimizer, X_val=None, y_val=None, initialize_weights=True, logging=True, train_dataset=None): # optimizer contains hyperparameters
    """
    Fit an MLP with backprop and minibatch GD

    Inputs:
    X:                    input features, shape (N, D)
    y:                    labels, shape (N,)
    C:                    number of classes
    optimizer:            holds nb_epochs, batch_size, learn_rate, and the update rule optimizer.run
    X_val, y_val:         optional validation data, evaluated at the end of each epoch
    initialize_weights:   if True, start with fresh weights, otherwise continue training
    logging:              prints a progress line per epoch
    train_dataset:        dataset used to generate a newly augmented set at start of each epoch. none by default indicates no augmenting

    Output:
    stats:  dict of per-epoch lists (train_loss, train_acc, val_loss, val_acc, elapsed)
            where 'time' is cumulative training time so-far
    """
    N, D = X.shape
    if initialize_weights:
      self.init_weights(D, C) # start with fresh weight

    rng = np.random.default_rng(SEED) # for shuffling

    stats = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'elapsed': []}
    time_taken = 0.0

    def _produce_batches(N, batch_size, rng):
      shuffle = rng.permutation(N) # shuffle
      for i in range(0, N, batch_size): # step by batch_size until N
        yield shuffle[i:i+batch_size]

    for epoch in range(optimizer.nb_epochs):
      time_start = time.perf_counter()

      # check if augmenting
      if train_dataset is not None:
        X_epoch, y_epoch = to_arrays(train_dataset)
      else:
        X_epoch, y_epoch = X, y

      for batch in _produce_batches(len(X_epoch), optimizer.batch_size, rng):
        Xs, ys = X_epoch[batch], y_epoch[batch] # matrices of shape (batch_size, D), (batch_size,)

        # forward pass + softmax
        logits, a_prevs = self.forward(Xs, training=True)
        p = softmax(logits) # probabilities output

        # compute gradients with backpropagation, note regularization is done here
        dW, db = self.backprop(p, ys, a_prevs)

        # step forward in minibatch SGD
        optimizer.run(self, dW, db)

      time_taken += time.perf_counter() - time_start # time for each epoch

      # collect metrics
      validating = X_val is not None and y_val is not None
      train_loss, train_acc = self.eval(X, y)
      stats['train_loss'].append(train_loss)
      stats['train_acc'].append(train_acc)
      stats['elapsed'].append(time_taken)
      if validating:
        validate_loss, validate_acc = self.eval(X_val, y_val)
        stats['val_loss'].append(validate_loss)
        stats['val_acc'].append(validate_acc)

      if logging and (epoch + 1) % 10 == 0: # print every 10th
        print(f"Epoch #{epoch+1}/{optimizer.nb_epochs}:  "
              f"loss {train_loss:.4f} | acc {train_acc:.4f}")
        if validating:
          print(f"loss {validate_loss:.4f} | acc {validate_acc:.4f}")

    return stats


  def predict(self, X):
    """Returns a classification prediction of input X by the MLP neural network"""
    logits, _ = self.forward(X)
    return logits.argmax(axis=1) # argmax of softmax = highest probability class

  def eval(self, X, y):
    """
    Mean CE and accuracy on a full dataset

    Inputs:
    X:    inputs, shape (N, D)
    y:    labels, shape (N,)

    Output:
    (mean CE, accuracy): statistical measures pasing over the entire dataset (no regularization penalty)
    """
    logits,_ = self.forward(X)
    p = softmax(logits)

    CE = cross_entropy(y, p)
    acc = np.mean(logits.argmax(axis=1) == y) # measure of correct predictions

    return CE, acc

## Gradient Descent

Minibatch SGD

L1+L2 regularization

Dropout regularization

Gradient clipping

Momentum

In [ ]:
# Gradient Descent
# ________________

class GradientDescent:

  def __init__(self, lr=0.01, batch_size=128, nb_epochs=50, clip_threshold=None, momentum=0.0):
    """
    Minibatch stochastic gradient descent. Holds hyperparameters and update method,
    while MLP.fit runs epoch-on-batch loop and calls run() once per batch

    Inputs:
    lr:             learning rate / step size
    batch_size:     number of samples in a minibatch (1 = pure stochastic, N = full batch GD)
    nb_epochs:      number of training set passes
    clip_threshold: max size tolerated until rescaling gradients, to prevent gradient explosion
    """
    self.lr = lr
    self.batch_size = batch_size
    self.nb_epochs = nb_epochs
    self.clip_threshold = clip_threshold
    self.prev_gradient_norm = None # previously computed gradient norm for clipping logging
    self.momentum = momentum
    self.velW, self.velb = None, None # velocities for momentum

  def reset_velocities(self):
    self.velW, self.velb = None, None

  def _compute_norm(self, dW, db):
    total = 0.0
    for gradient in dW + db: # list concat
      total += np.sum(gradient ** 2) # sum of squares
    return np.sqrt(total) # norm

  def _clip(self, dW, db):
    norm = self._compute_norm(dW, db)
    self.prev_gradient_norm = norm

    if norm > self.clip_threshold:
      dW = [gradient * (self.clip_threshold / norm) for gradient in dW]
      db = [gradient * (self.clip_threshold / norm) for gradient in db]

    return dW, db

  def run(self, model: MLP, dW, db):
    """
    Apply a single updateto model.weights and model.biases
    Includes rolling momentum for faster convergence!
    """
    if self.clip_threshold is not None:
      dW, db = self._clip(dW, db)

    if self.velW is None or self.velb is None:
      self.velW = [np.zeros_like(w) for w in model.weights]
      self.velb = [np.zeros_like(b) for b in model.biases]

    for i in range(model.nb_layers):
      # momentum gathering
      self.velW[i] = self.momentum * self.velW[i] + dW[i]
      self.velb[i] = self.momentum * self.velb[i] + db[i]

      # weight updates, by momentum instead of raw gradients
      model.weights[i] -= self.lr * self.velW[i]
      model.biases[i] -= self.lr * self.velb[i]

In [ ]:
# Tests!
# ______

# check shapes after init
model = MLP(hidden=(256,256))
model.init_weights(784, 10)
print([w.shape for w in model.weights])

model_no_layers = MLP(hidden=())
model_no_layers.init_weights(784, 10)
print([w.shape for w in model_no_layers.weights])

# forward pass + loss at initialization
X_sample = np.random.default_rng(0).normal(size=(64, 784))
y_sample = np.random.default_rng(1).integers(0, 10, size=64)
logits, a_prevs = model.forward(X_sample)
p = softmax(logits)
print(logits.shape, np.allclose(p.sum(axis=1), 1)) # should be (64, 10) and True. allclose checks all are 1

# cross-entropy on uniform probabilities should be roughly ln(10)
# probability of guessing right = 1/10
# nll = -ln(1/10) ~= -2.303 loss
print(cross_entropy(y_sample, np.full((64, 10), 0.1)))

# backpropagation outputs and shapes
dW, db = model.backprop(p, y_sample, a_prevs)
print(type(dW), len(dW), type(db), len(db))
print([g.shape for g in dW])
print([w.shape for w in model.weights]) # should agree with gradients
print([g.shape for g in db])
print([b.shape for b in model.biases])  # should agree with gradients

# way overfit test
Xs, ys = X_train[:100], y_train[:100]   # from normalized global arrays
model_two_elec_boo = MLP(hidden=(256,))
stats = model_two_elec_boo.fit(Xs, ys, 10, GradientDescent(lr=0.1, batch_size=20, nb_epochs=100), logging=True)
print(f"Before train: {stats['train_loss'][0]}, After train: {stats['train_loss'][-1]}, Acc: {stats['train_acc'][-1]}") # loss falling to near 0, acc at practically 1.0


# prediction tests
model_three_more_4_me = MLP(hidden=(256, 256))
model_three_more_4_me.init_weights(784, 10)

pred = model_three_more_4_me.predict(X_val)
print(np.array_equal(model_three_more_4_me.predict(X_val), model_three_more_4_me.predict(X_val))) # should be the same, true
logits, _ = model_three_more_4_me.forward(X_val)
print(np.array_equal(pred, softmax(logits).argmax(axis=1)))  # should be consistent with softmax, true

# check predictions beat chance
optimizer = GradientDescent(lr=0.1, batch_size=128, nb_epochs=5)
stats = model_three_more_4_me.fit(X_train, y_train, 10, optimizer, X_val, y_val, logging=False)
pred = model_three_more_4_me.predict(X_test)
print(evaluate_acc(y_test, pred))       # roughly 0.85 for 5 epochs, yay!
print(np.bincount(pred, minlength=10))  # every class predicted, about 1000 each?

# momentum check
another_model = MLP(hidden=(256,256))
without_momentum = GradientDescent(lr=0.01, batch_size=128, nb_epochs=15, momentum=0.0)
with_momentum = GradientDescent(lr=0.01, batch_size=128, nb_epochs=15, momentum=0.9)
another_model.init_weights(784, 10)
stats_with = another_model.fit(X_train, y_train, 10, with_momentum, X_val, y_val, logging=False)
another_model.init_weights(784, 10) # reset weights
stats_without = another_model.fit(X_train, y_train, 10, without_momentum, X_val, y_val, logging=False)

print(f"Time to converge without momentum: {np.sum(stats_without['elapsed'])}")
print(f"Time to converge with momentum: {np.sum(stats_with['elapsed'])}")

In [ ]:
# functionality tests by creating training curves; comparing activation functions

EPOCHS = 60

configs = {
  'ReLU': dict(hidden=(256, 256), a_func='relu'),
  'tanh': dict(hidden=(256, 256), a_func='tanh'),
}

# train loss and validation loss curves
for METRIC in ('train_loss', 'val_loss'):
  runs = {}
  for label, cfg in configs.items():
    model = MLP(**cfg, seed=SEED)
    opt = GradientDescent(lr=0.1, batch_size=128, nb_epochs=EPOCHS)
    print(label)
    runs[label] = model.fit(X_train, y_train, 10, opt, X_val, y_val, logging=False)

  fig, (ax_ep, ax_t) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

  for i, (label, s) in enumerate(runs.items()):
    color = f'C{i}'
    y = s[METRIC]
    epochs = np.arange(1, len(y) + 1)

    for ax, x in ((ax_ep, epochs), (ax_t, s['elapsed'])):
      ax.plot(x, y, color=color, linewidth=2)
      ax.annotate(label, xy=(x[-1], y[-1]), xytext=(6, 0),
                  textcoords='offset points', color=color, va='center')

  for ax, xlabel in ((ax_ep, 'epoch #'), (ax_t, 'cumulative training time (s)')):
    ax.set_xlabel(xlabel)
    ax.margins(x=0.15) # room on the right for the line labels
    ax.grid(alpha=0.3)

  ax_ep.set_ylabel(METRIC.replace('_', ' '))
  plt.tight_layout()
  plt.show()


In [ ]:
# functionality tests by creating training curves; comparing regularization methods

EPOCHS = 60

configs = {
  'no reg': dict(hidden=(256, 256), a_func='relu'),
  'L1':     dict(hidden=(256, 256), a_func='relu', reg='L1', lam=1e-4),
  'L2':     dict(hidden=(256, 256), a_func='relu', reg='L2', lam=1e-3),
  'dropout 0.3':  dict(hidden=(256, 256), a_func='relu', dropout=0.3),
  'dropout 0.5':  dict(hidden=(256, 256), a_func='relu', dropout=0.5), # the standard

}

runs = {}
for label, cfg in configs.items():
  model = MLP(**cfg, seed=SEED)
  opt = GradientDescent(lr=0.1, batch_size=128, nb_epochs=EPOCHS)
  print(label)
  runs[label] = model.fit(X_train, y_train, 10, opt, X_val, y_val, logging=False)

#fig, (ax_ep, ax_t) = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
fig, ax_ep = plt.subplots(figsize=(7, 5))
fig.suptitle('Regularization: Training vs. Validation Loss', fontsize=14, y=1.00)


for i, (label, s) in enumerate(runs.items()):
  color = f'C{i}'
  epochs = np.arange(1, len(s['val_loss']) + 1)

  #for ax, x in ((ax_ep, epochs), (ax_t, s['elapsed'])):
  ax = ax_ep
  x = epochs
  ax.plot(x, s['train_loss'], color=color, linestyle='--', linewidth=1.5)  # train
  ax.plot(x, s['val_loss'], color=color, linewidth=2)                      # validation
  #ax.annotate(label, xy=(x[-1], s['val_loss'][-1]), xytext=(6, 0), textcoords='offset points', color=color, va='center')

config_handles = [ # for nice labelling that doesnt overlap
    plt.Line2D([0], [0], color=f'C{i}', linewidth=2, label=label)
    for i, label in enumerate(runs)
]
style_handles = [
    plt.Line2D([0], [0], color='black', linestyle='--', label='Training'),
    plt.Line2D([0], [0], color='black', linestyle='-', label='Validation')
]
fig.legend(handles=config_handles, loc='lower center', bbox_to_anchor=(0.5, 0.02), ncol=5, fontsize=9, frameon=False)
fig.legend(handles=style_handles, loc='lower center', bbox_to_anchor=(0.5, 0.10), ncol=2, fontsize=9, frameon=False)

# for ax, xlabel in ((ax_ep, 'epoch'), (ax_t, 'cumulative training time (s)')):
#   ax.set_xlabel(xlabel)
#   #ax.margins(x=0.15) # room on the right for the line labels
#   ax.grid(alpha=0.3)

ax_ep.set_xlabel('Epoch')
ax_ep.grid(alpha=0.3)

ax_ep.set_ylabel('Cross-entropy Loss')
fig.subplots_adjust(bottom=0.28) # give space between plots and sublegend
plt.show()

## Hyperparameter Tuning w/ Scikit Learn

We need to wrap the MLP implementation into a scikit-learn compliant bundle, that way we can use their hyperparameter tuning validation library easily without having to implement our own

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import GridSearchCV, PredefinedSplit
import pandas as pd
# https://scikit-learn.org/stable/modules/grid_search.html

class Wrapper(BaseEstimator, ClassifierMixin):
  def __init__(self, hidden=(256, 256), a_func='relu', reg=None, lam=0.0, dropout=0.0, lr=0.1, batch_size=128, nb_epochs=10):
    # scikitlearn rules says the values can't change in the initializer and they have to have the same names as arguments
    self.hidden, self.a_func, self.reg, self.lam, self.dropout = hidden, a_func, reg, lam, dropout # regularization + config
    self.lr, self.batch_size, self.nb_epochs = lr, batch_size, nb_epochs                           # fitting + descent

  def fit(self, X, y):
    # node self.model_ is a private convention, like helpers are def _func()
    self.model_ = MLP(hidden=self.hidden, a_func=self.a_func, reg=self.reg, lam=self.lam, dropout=self.dropout, seed=SEED)
    optimizer = GradientDescent(lr=self.lr, batch_size=self.batch_size, nb_epochs=self.nb_epochs)
    self.model_.fit(X, y, 10, optimizer, logging=False)
    self.classes_ = np.unique(y) # returns sorted unique classes in an array

    return self # skl expectation is that fit returns self

  def predict(self, X):
    return self.model_.predict(X)

X_full = np.concatenate([X_train, X_val])
y_full = np.concatenate([y_train, y_val])
test_fold = np.concatenate([np.full(len(X_train), -1), np.zeros(len(X_val))]) # not going with k-folds

grid = {
    'lr': [0.01, 0.05, 0.1],
    'batch_size': [64, 128, 256],
    'nb_epochs': [15] # got worse after 10
}

search = GridSearchCV(Wrapper(), grid, cv=PredefinedSplit(test_fold), refit=False, verbose=2)
search.fit(X_full, y_full)

results = pd.DataFrame(search.cv_results_)
print(results[['params', 'mean_test_score', 'mean_fit_time']].sort_values('mean_test_score', ascending=False))
print(search.best_params_, search.best_score_)

#T3: Experiments and Results

## Setup + Helpers for Q1-5

In [ ]:
def report_runs(runs, use_sublegend=True, title=None):
  """
  Plots training and validation accuracy for multiple runs.

  Parameters:
  runs: A list of (base_label, stats, test_acc), where stats is an output from MLP.fit()
  use_sublegend: Boolean, if True, uses custom split legend at the bottom
  """
  fig = plt.gcf() if plt.get_fignums() else plt.figure(figsize=(6, 4.5))

  for i in range(len(runs)):
    base_label, stats, test_acc = runs[i]
    color = f"C{i}"

    plt.plot(stats['train_acc'], label=base_label+", train", color=color)
    plt.plot(stats['val_acc'], "--", label=base_label+", validation", color=color)

    print(f"{base_label} Training Accuracy: {stats['train_acc'][-1]:.4f}")
    print(f"{base_label} Validation Accuracy: {stats['val_acc'][-1]:.4f}")
    print(f"{base_label} Test Accuracy: {test_acc:.4f}")

  plt.xlabel("Epochs")
  plt.ylabel("Accuracy")
  plt.grid(alpha=0.3)

  # compact legend below for readability
  if use_sublegend:
    config_handles = [
        plt.Line2D([0], [0], color=f'C{i}', linewidth=2, label=label)
        for i, (label, _, _) in enumerate(runs)
    ]
    style_handles = [
        plt.Line2D([0], [0], color='black', linestyle='-', label='Training'),
        plt.Line2D([0], [0], color='black', linestyle='--', label='Validation')
    ]
    fig.legend(handles=config_handles, loc='lower center', bbox_to_anchor=(0.5, 0.02), ncol=len(runs), fontsize=9, frameon=False)
    fig.legend(handles=style_handles, loc='lower center', bbox_to_anchor=(0.5, 0.10), ncol=2, fontsize=9, frameon=False)
    fig.subplots_adjust(bottom=0.28)
  else:
    plt.legend()

  if title is not None:
    plt.title(label=title)

  plt.show()


def multi_train_val_time_plot(runs, use_sublegend=False, title=None):
  """
  Plots training and validation accuracy for multiple runs against cumulative elapsed time.

  Parameters:
  runs: A list of (base_label, stats), where stats is an output from MLP.fit()
  use_sublegend: Boolean, if True, uses custom split legend at the bottom
  """
  fig = plt.figure(figsize=(6, 4.5))
  for i, (base_label, stats, test_acc) in enumerate(runs):
    color = f"C{i}"
    # stats['elapsed'] has cumulative training time per epoch
    plt.plot(stats['elapsed'], stats['train_acc'], label=base_label+", train", color=color)
    plt.plot(stats['elapsed'], stats['val_acc'], "--", label=base_label+", validation", color=color)

  plt.xlabel("Cumulative elapsed time (s)")
  plt.ylabel("Accuracy")
  plt.grid(alpha=0.3)

  # compact legend below for readability
  if use_sublegend:
    config_handles = [
        plt.Line2D([0], [0], color=f'C{i}', linewidth=2, label=label)
        for i, (label, _, _) in enumerate(runs)
    ]
    style_handles = [
        plt.Line2D([0], [0], color='black', linestyle='-', label='Training'),
        plt.Line2D([0], [0], color='black', linestyle='--', label='Validation')
    ]
    fig.legend(handles=config_handles, loc='lower center', bbox_to_anchor=(0.5, 0.02), ncol=len(runs), fontsize=9, frameon=False)
    fig.legend(handles=style_handles, loc='lower center', bbox_to_anchor=(0.5, 0.10), ncol=2, fontsize=9, frameon=False)
    fig.subplots_adjust(bottom=0.28)
  else:
    plt.legend()

  if title is not None:
    plt.title(label=title)

  plt.show()

## Q1

In [ ]:
no_hidden = MLP(hidden=())
one_hidden = MLP(hidden=(256,), a_func='relu')
two_hidden = MLP(hidden=(256, 256), a_func='relu')

In [ ]:
runs_q1 = []

opt = GradientDescent()
runs_q1.append(("0 hidden layers" ,
                no_hidden.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val),
                evaluate_acc(y_test, no_hidden.predict(X_test))))

opt = GradientDescent()
runs_q1.append(("1 hidden layer",
                one_hidden.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val),
               evaluate_acc(y_test, one_hidden.predict(X_test))))

opt = GradientDescent()
runs_q1.append(("2 hidden layers",
                two_hidden.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val),
                evaluate_acc(y_test, two_hidden.predict(X_test))))

In [ ]:
report_runs(runs_q1, use_sublegend=True, title="Effect of network depth on accuracy")

## Q2

In [ ]:
two_hidden_tanh = MLP(hidden=(256, 256), a_func='tanh')
two_hidden_leaky = MLP(hidden=(256, 256), a_func='leaky_relu', leaky_factor=0.01)

In [ ]:
runs_q2 = []

opt = GradientDescent()
runs_q2.append(("ReLU",
                two_hidden.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val),
                evaluate_acc(y_test, two_hidden.predict(X_test))))

opt = GradientDescent()
runs_q2.append(("tanh",
                two_hidden_tanh.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val),
                evaluate_acc(y_test, two_hidden_tanh.predict(X_test))))

opt = GradientDescent()
runs_q2.append(("Leaky ReLU",
                two_hidden_leaky.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val),
                evaluate_acc(y_test, two_hidden_leaky.predict(X_test))))

In [ ]:
report_runs(runs_q2, title="Effect of activation choice on accuracy")

## Q3

In [ ]:
two_hidden_l1 = MLP(hidden=(256, 256), a_func='relu', reg='L1', lam=5e-4)
two_hidden_l2 = MLP(hidden=(256, 256), a_func='relu', reg='L2', lam=5e-3)
two_hidden_dropout = MLP(hidden=(256, 256), a_func='relu', dropout = 0.3)

In [ ]:
runs_q3 = []

opt = GradientDescent()
runs_q3.append(('No Reg',
                two_hidden.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val),
                evaluate_acc(y_test, two_hidden.predict(X_test))))

opt = GradientDescent()
runs_q3.append((r'L1, $\lambda = 5x10^{-4}$',
                two_hidden_l1.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val),
                evaluate_acc(y_test, two_hidden_l1.predict(X_test))))

opt = GradientDescent()
runs_q3.append((r'L2, $\lambda = 5x10^{-3}$',
                two_hidden_l2.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val),
                evaluate_acc(y_test, two_hidden_l2.predict(X_test))))

opt = GradientDescent()
runs_q3.append(('Dropout, 0.3',
                two_hidden_dropout.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val),
                evaluate_acc(y_test, two_hidden_dropout.predict(X_test))))

In [ ]:
report_runs(runs_q3, use_sublegend=True, title="Methods for Regularization")

## Optional: Momentum

In [ ]:
runs_momentum = []

# no momentum
mlp_no_mom = MLP(hidden=(256, 256), a_func='relu', seed=SEED)
opt_no_mom = GradientDescent(momentum=0.0)
runs_momentum.append(("Momentum = 0.0",
                      mlp_no_mom.fit(X_train, y_train, 10, opt_no_mom, X_val=X_val, y_val=y_val,),
                      evaluate_acc(y_test, mlp_no_mom.predict(X_test))))

# some momentum
mlp_med_mom = MLP(hidden=(256, 256), a_func='relu', seed=SEED)
opt_med_mom = GradientDescent(momentum=0.5)
runs_momentum.append(("Momentum = 0.5",
                      mlp_med_mom.fit(X_train, y_train, 10, opt_med_mom, X_val=X_val, y_val=y_val),
                      evaluate_acc(y_test, mlp_med_mom.predict(X_test))))

# high momentum
mlp_high_mom = MLP(hidden=(256, 256), a_func='relu', seed=SEED)
opt_high_mom = GradientDescent(momentum=0.9)
runs_momentum.append(("Momentum = 0.9",
                      mlp_high_mom.fit(X_train, y_train, 10, opt_high_mom, X_val=X_val, y_val=y_val),
                      evaluate_acc(y_test, mlp_high_mom.predict(X_test))))

In [ ]:
report_runs(runs_momentum, use_sublegend=True, title="Impact of Momentum on Training Accuracy (Epochs)")
multi_train_val_time_plot(runs_momentum, use_sublegend=True, title="Impact of Momentum on Training Accuracy (Time)")

## Q4

In [ ]:
X_train_unnorm, y_train_unnorm, X_val_unnorm, y_val_unnorm, X_test_unnorm, y_test_unnorm = load_data(normalize=False)

In [ ]:
runs_q4 = []
# momentum and clipping added to prevent explosion on unnormalized data
opt = GradientDescent(lr=10**(-4), clip_threshold=10, momentum=0.9)
runs_q4.append(('Normalized',
                two_hidden.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val),
                evaluate_acc(y_test, two_hidden.predict(X_test))))

opt = GradientDescent(lr=10**(-4), clip_threshold=10, momentum=0.9)
runs_q4.append(('Unnormalized',
                two_hidden.fit(X_train_unnorm, y_train_unnorm, 10, opt, X_val=X_val_unnorm, y_val=y_val_unnorm),
                evaluate_acc(y_test_unnorm, two_hidden.predict(X_test_unnorm))))

In [ ]:
report_runs(runs_q4, title="Effect of normalization on accuracy")

## Q5

In [ ]:
runs_q5 = []

opt = GradientDescent(nb_epochs=60)
runs_q5.append(('No augmentation',
                two_hidden.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val),
                evaluate_acc(y_test, two_hidden.predict(X_test))))

opt = GradientDescent(nb_epochs=60)
runs_q5.append(('Augmentation',
                two_hidden.fit(X_train, y_train, 10, opt, X_val=X_val, y_val=y_val, train_dataset=aug_train),
                evaluate_acc(y_test, two_hidden.predict(X_test))))

In [ ]:
report_runs(runs_q5, title="Effect of data augmentation on accuracy")

## Setup + Helpers for Q6-10

In [ ]:
# Shared helpers for questions 6 to 10
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from torch.utils.data import TensorDataset
from torchvision import models

EPOCHS_CNN = 50  # Same as MLP from Q1-Q5
EPOCHS_PRE = 50
BATCH = 128
WORKERS = 2
torch.manual_seed(SEED) # 1234

# Reference numbers for best MLP from Q5
MLP_TEST_ACC = evaluate_acc(y_test, two_hidden.predict(X_test))
MLP_TRAIN_TIME = runs_q5[-1][1]['elapsed'][-1]
MLP_VAL_ACC = runs_q5[-1][1]['val_acc'][-1]
print(f"Best MLP (Q5): test acc {MLP_TEST_ACC:.4f}, val acc {MLP_VAL_ACC:.4f}, train time {MLP_TRAIN_TIME:.1f}s")

def sync():
  if device.type == 'cuda': torch.cuda.synchronize()
  elif device.type == 'mps': torch.mps.synchronize()


def build_sets(ds_cls, tf_eval, tf_aug=None, val_frac=VAL_SPLIT):
  """
  Same split as the MLP section (default_rng(SEED).permutation), so FashionMNIST reproduces train_idx/val_idx.
  Returns clean train, clean val, test, augmented train (None if tf_aug is None), train_idx, val_idx

  Most useful in Q9
  """
  full_eval = ds_cls('./data', train=True, download=True, transform=tf_eval) # whole training file with clean transform
  test = ds_cls('./data', train=False, download=True, transform=tf_eval)
  perm = np.random.default_rng(SEED).permutation(len(full_eval))
  n_val = int(val_frac * len(full_eval)) # number of validation samples (10%)
  v_idx, t_idx = perm[:n_val], perm[n_val:]
  aug = None # default
  if tf_aug is not None:  # only build it when an augmentation pipeline is given
    aug = Subset(ds_cls('./data', train=True, download=True, transform=tf_aug), t_idx)  # augmented copy, training indices only
  return Subset(full_eval, t_idx), Subset(full_eval, v_idx), test, aug, t_idx, v_idx  # validation is never augmented


def cache_tensors(ds, bs=1000):
  """
  Run transform pipeline once and keep the tensors, so clean sets aren't re-decoded each epoch

  Helps speed up runs.
  """
  xs, ys = [], []
  for xb, yb in DataLoader(ds, batch_size=bs, shuffle=False, num_workers=WORKERS):
    xs.append(xb); ys.append(yb)
  return TensorDataset(torch.cat(xs), torch.cat(ys)) # glue batches together into 1 dataset


def loader(ds, shuffle=False, bs=BATCH, workers=0): # shorthand for creating DataLoader
  return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=workers)


def evaluate_torch(model, dl): # mean loss and accuracy of a model over a whole loader
  model.eval()
  loss_sum, correct, n = 0.0, 0, 0
  with torch.no_grad():
    for xb, yb in dl:
      xb, yb = xb.to(device), yb.to(device)
      out = model(xb)
      loss_sum += F.cross_entropy(out, yb, reduction='sum').item()
      correct += (out.argmax(1) == yb).sum().item()
      n += len(yb)
  return loss_sum / n, correct / n


def train_torch(model, train_dl, eval_dls, epochs, lr=0.05, momentum=0.9, weight_decay=0.0, eval_model=None, logging=True):
  model.to(device)
  eval_model = model if eval_model is None else eval_model.to(device)
  params = [p for p in model.parameters() if p.requires_grad]  # frozen layers aren't updated
  opt = torch.optim.SGD(params, lr=lr, momentum=momentum, weight_decay=weight_decay)
  stats = {k: [] for k in ('train_loss', 'train_acc', 'val_loss', 'val_acc', 'test_loss', 'test_acc', 'elapsed')}
  total = 0.0
  for epoch in range(epochs):
    model.train()
    sync(); t0 = time.perf_counter()
    for xb, yb in train_dl:
      xb, yb = xb.to(device), yb.to(device)
      opt.zero_grad()
      loss = F.cross_entropy(model(xb), yb)  # softmax is inside the cross-entropy
      loss.backward()
      opt.step()
    sync(); total += time.perf_counter() - t0
    stats['elapsed'].append(total)
    for name, dl in eval_dls.items():
      l, a = evaluate_torch(eval_model, dl)
      stats[f'{name}_loss'].append(l); stats[f'{name}_acc'].append(a)
    if logging and (epoch + 1) % 5 == 0:
      print(f"Epoch #{epoch+1}/{epochs}: train acc {stats['train_acc'][-1]:.4f} | val acc {stats['val_acc'][-1]:.4f} | test acc {stats['test_acc'][-1]:.4f}")
  return stats


# plot_runs and summarize for Q10
def plot_runs(runs, title=''):
  """
  runs: list of (label, stats)
  Left: train (solid) / val (dashed) / test (dotted) accuracy vs epoch
  Right: test acc vs cumulative train time
  """
  fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
  for i, (label, s) in enumerate(runs):
    c = f'C{i}'
    ep = np.arange(1, len(s['train_acc']) + 1)
    ax1.plot(ep, s['train_acc'], color=c, label=f'{label}, train')
    ax1.plot(ep, s['val_acc'], '--', color=c, label=f'{label}, validation')
    if s.get('test_acc'): ax1.plot(ep, s['test_acc'], ':', color=c, label=f'{label}, test')
    ax2.plot(s['elapsed'], s['test_acc'] if s.get('test_acc') else s['val_acc'], color=c, label=label)
  ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.grid(alpha=0.3); ax1.legend(loc="lower right", fontsize=7)
  ax2.set_xlabel('Cumulative training time (s)'); ax2.set_ylabel('Test accuracy'); ax2.grid(alpha=0.3); ax2.legend(loc="lower right", fontsize=8)
  fig.suptitle(title); plt.tight_layout(); plt.show()


def summarize(runs):
  """Test accuracy is read at the epoch with the best VALIDATION accuracy"""
  rows = []
  for label, s in runs:
    b = int(np.argmax(s['val_acc']))
    rows.append({'model': label, 'best epoch': b + 1, 'val acc': s['val_acc'][b], 'test acc': s['test_acc'][b],
                 'final train acc': s['train_acc'][-1], 'train time (s)': s['elapsed'][-1],
                 's/epoch': s['elapsed'][-1] / len(s['elapsed'])})
  return pd.DataFrame(rows).round(4)

## Q6

In [ ]:
norm_fm = transforms.Normalize((MEAN,), (STD,)) # standardize with the training-split mean/std
tf_eval_fm = transforms.Compose([transforms.ToTensor(), norm_fm]) # PIL to tensor in [0,1] to standardized
tf_aug_fm = transforms.Compose(build_augmentations() + [transforms.ToTensor(), norm_fm])

fm_train, fm_val, fm_test, fm_train_aug, t_i, v_i = build_sets(datasets.FashionMNIST, tf_eval_fm, tf_aug_fm)
assert np.array_equal(t_i, train_idx) and np.array_equal(v_i, val_idx)  # same split as the MLPs

fm_train_c, fm_val_c, fm_test_c = cache_tensors(fm_train), cache_tensors(fm_val), cache_tensors(fm_test)  # decode clean sets once
fm_eval_dls = {'train': loader(fm_train_c, bs=1000), 'val': loader(fm_val_c, bs=1000), 'test': loader(fm_test_c, bs=1000)}  # evaluation loaders
print(fm_train_c.tensors[0].shape, fm_train_c.tensors[0].mean().item(), fm_train_c.tensors[0].std().item())  # expect (54000,1,28,28), ~0, ~1


class CNN(nn.Module): # configurable size
  """conv -> ReLU -> pool -> conv -> ReLU -> pool -> FC(256) -> ReLU -> FC(10).  Output = logits"""
  def __init__(self, in_ch=1, img=28, c1=32, c2=64, k=3, hidden=256, n_classes=10):
    super().__init__()
    self.conv1 = nn.Conv2d(in_ch, c1, k, padding=k // 2) # first conv layer
    self.conv2 = nn.Conv2d(c1, c2, k, padding=k // 2) # second conv layer
    self.pool = nn.MaxPool2d(2) # 2x2 max pooling halves height and width
    self.fc1 = nn.Linear(c2 * (img // 4) ** 2, hidden) # FC hidden layer
    self.fc2 = nn.Linear(hidden, n_classes) # FC output layer (one logit per class)

  def forward(self, x): # forward pass
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    return self.fc2(F.relu(self.fc1(x.flatten(1))))  # flatten, FC1 to ReLU, FC2 to logits (no softmax cause loss adds it)

In [ ]:
# short runs, hyperparameters selected on val accuracy
SWEEP_EPOCHS = 8
rows = [] # results of the lr sweep
for lr in (0.005, 0.01, 0.05, 0.1): # candidate lr's
  torch.manual_seed(SEED)
  s = train_torch(CNN(), loader(fm_train_c, shuffle=True), fm_eval_dls, SWEEP_EPOCHS, lr=lr, logging=False)  # train fresh CNN
  rows.append({'sweep': 'lr', 'value': lr, 'val acc': max(s['val_acc']), 'time (s)': s['elapsed'][-1]})
sweep_lr = pd.DataFrame(rows); print(sweep_lr) #show table
BEST_LR = float(sweep_lr.sort_values('val acc').iloc[-1]['value'])

rows = []
for (c1, c2) in ((16, 32), (32, 64), (64, 128)): # channel counts of the two conv layers
  for k in (3, 5): # kernel sizes
    torch.manual_seed(SEED)
    s = train_torch(CNN(c1=c1, c2=c2, k=k), loader(fm_train_c, shuffle=True), fm_eval_dls, SWEEP_EPOCHS, lr=BEST_LR, logging=False)
    rows.append({'channels': f'{c1}-{c2}', 'kernel': k, 'val acc': max(s['val_acc']), 'time (s)': s['elapsed'][-1]})
sweep_arch = pd.DataFrame(rows); print(sweep_arch) # show table
# thes tables in the appendix
# chosen lr, channels, and kernel in text.
BEST_C1, BEST_C2, BEST_K = 32, 64, 3

In [ ]:
torch.manual_seed(SEED)
cnn_q6 = CNN(c1=BEST_C1, c2=BEST_C2, k=BEST_K)
stats_cnn = train_torch(cnn_q6, loader(fm_train_c, shuffle=True), fm_eval_dls, EPOCHS_CNN, lr=BEST_LR) # train on clean (ie non augmented) data

runs_q6 = [('CNN', stats_cnn)] # list of (label, stats) for plotting
plot_runs(runs_q6, 'Q6: CNN on FashionMNIST') # train/val/test curves
summ = summarize(runs_q6)
summ.loc[len(summ)] = ['Best MLP (Q5, augmented)', np.nan, MLP_VAL_ACC, MLP_TEST_ACC, np.nan, MLP_TRAIN_TIME, np.nan]  # add MLP row for comparison
print(summ)
# since MLP is numPy on CPU and CNN runs on local device (gpu), we need to compare accuracy and focus less on time.

## Q7

In [ ]:
torch.manual_seed(SEED)
cnn_q7 = CNN(c1=BEST_C1, c2=BEST_C2, k=BEST_K) # fresh CNN with identical architecture
stats_cnn_aug = train_torch(cnn_q7, loader(fm_train_aug, shuffle=True, workers=WORKERS), fm_eval_dls, EPOCHS_CNN, lr=BEST_LR) # train on augmented images

runs_q7 = [('CNN, no augmentation', stats_cnn), ('CNN, augmentation', stats_cnn_aug)]
plot_runs(runs_q7, 'Q7: CNN with vs. without data augmentation') # curves
print(summarize(runs_q7)) # accuracy and speed table
# train time here includes CPU side PIL augmentation (the dataloader is the bottleneck), which is the 'speed' cost

## Q8

In [ ]:
IMN_MEAN, IMN_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225] # per-channel statistics ResNet was trained with (from models.ResNet18_Weights.DEFAULT.transforms())
RES = 64 # images are upscaled to 64x64
norm_imn = transforms.Normalize(IMN_MEAN, IMN_STD)
tf_eval_pre = transforms.Compose([transforms.Resize(RES), transforms.Grayscale(3), transforms.ToTensor(), norm_imn]) # resize, copy gray to 3 channels, tensor, normalize
tf_aug_pre = transforms.Compose(build_augmentations() + [transforms.Resize(RES), transforms.Grayscale(3), transforms.ToTensor(), norm_imn])


def make_backbone(): # frozen ResNet feature extractor
  m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT) # load ImageNet pre-trained weights (downloads on first use)
  m.fc = nn.Identity()
  for p in m.parameters():
    p.requires_grad = False # freeze it
  return m.to(device).eval() # move to device; eval mode so BatchNorm uses its stored statistics


@torch.no_grad() # no gradients needed while extracting features
def extract_features(backbone, ds, bs=500): # run images through the frozen backbone once
  backbone.eval() # make sure BatchNorm is frozen
  fs, ys = [], [] # feature batches and labels
  for xb, yb in DataLoader(ds, batch_size=bs, shuffle=False, num_workers=WORKERS):
    fs.append(backbone(xb.to(device)).cpu()); ys.append(yb)
  return TensorDataset(torch.cat(fs), torch.cat(ys)) # features + labels as a dataset


def make_head(n_hidden, width=256, in_dim=512, n_classes=10): # trainable fully connected classifier
  layers, d = [], in_dim # list of layers; d is current feature size
  for _ in range(n_hidden): # one block per hidden layer
    layers += [nn.Linear(d, width), nn.ReLU()]; d = width # Linear to ReLU
  layers.append(nn.Linear(d, n_classes)) # logits
  return nn.Sequential(*layers)


class Pretrained(nn.Module):
  """Frozen backbone + trainable head. train() keeps the backbone in eval mode"""
  def __init__(self, backbone, head): # takes the two parts
    super().__init__()
    self.backbone, self.head = backbone, head

  def train(self, mode=True): # model.train() must not un-freeze BatchNorm
    super().train(mode)
    self.backbone.eval()
    return self

  def forward(self, x): # forward pass
    with torch.no_grad():
      f = self.backbone(x).flatten(1)
    return self.head(f)


backbone = make_backbone() # build the frozen ResNet
pre_train, pre_val, pre_test, pre_train_aug, _, _ = build_sets(datasets.FashionMNIST, tf_eval_pre, tf_aug_pre)
t0 = time.perf_counter()
F_train, F_val, F_test = (extract_features(backbone, d) for d in (pre_train, pre_val, pre_test)) # cache clean features for the three splits
print(f"feature extraction: {time.perf_counter() - t0:.1f}s", F_train.tensors[0].shape)
feat_dls = {'train': loader(F_train, bs=1000), 'val': loader(F_val, bs=1000), 'test': loader(F_test, bs=1000)} # loaders over cached features

In [ ]:
# Experiment/Extra Something for the Project >:)
# "Justify your choice of how many fully connected layers that you have added to the pre-trained model through careful experiments"
# how many hidden FC layers (width 256) after the frozen backbone?  0 = a single linear layer on the features.
HEAD_EPOCHS, HEAD_LR = 20, 0.05 # training budget and learning rate for  head experiment
depth_runs, rows = [], [] # stats per depth and table rows
for n_hidden in (0, 1, 2, 3): # candidate depths
  torch.manual_seed(SEED)
  s = train_torch(make_head(n_hidden), loader(F_train, shuffle=True), feat_dls, HEAD_EPOCHS, lr=HEAD_LR, logging=False) # train head on cached features
  depth_runs.append((f'{n_hidden} hidden FC', s)) # keep full history for plotting
  b = int(np.argmax(s['val_acc'])) # epoch of best val acc
  rows.append({'n_hidden': n_hidden, 'val acc': s['val_acc'][b], 'final train acc': s['train_acc'][-1], # best val acc, last train acc
               'gap (train-val)': s['train_acc'][-1] - s['val_acc'][-1], 'time (s)': s['elapsed'][-1]}) # overfitting gap, training time
depth_table = pd.DataFrame(rows).round(4); print(depth_table) # table

# val accuracy vs number of FC layers. !this figure to justify choice in report
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(depth_table['n_hidden'], depth_table['val acc'], 'o-', label='validation') # val acc vs depth
ax1.plot(depth_table['n_hidden'], [r[1]['train_acc'][-1] for r in depth_runs], 's--', label='train') # train acc vs depth
ax1.set_xlabel('# hidden FC layers'); ax1.set_ylabel('Accuracy'); ax1.set_xticks(range(4)); ax1.grid(alpha=0.3); ax1.legend()
for i, (label, s) in enumerate(depth_runs): # one curve per depth
  ax2.plot(np.arange(1, HEAD_EPOCHS + 1), s['val_acc'], color=f'C{i}', label=label) # val acc per epoch
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Validation accuracy'); ax2.grid(alpha=0.3); ax2.legend()
plt.tight_layout(); plt.show() #fig

# the default is best val acc. !!If two are within noise, prefer the shallower one (cheaper, less overfitting)
BEST_DEPTH = int(depth_table.sort_values('val acc').iloc[-1]['n_hidden']) # depth with highest validation accuracy
print('chosen depth:', BEST_DEPTH)

In [ ]:
# final pre-trained model has frozen backbone + chosen head, trained w/ Q5 augmentation
# eval uses cached clean features through the head only (identical to running frozen backbone, but much cheaper).
torch.manual_seed(SEED)
head_q8 = make_head(BEST_DEPTH) # head with the chosen number of hidden layers from experiment
model_q8 = Pretrained(backbone, head_q8) # attach to frozen backbone
stats_pre = train_torch(model_q8, loader(pre_train_aug, shuffle=True, workers=WORKERS), feat_dls, EPOCHS_PRE,
                        lr=HEAD_LR, eval_model=head_q8) # evaluate with head on cached clean features

runs_q8 = [('Pre-trained ResNet-18 (frozen) + FC', stats_pre)] #for plotting
plot_runs(runs_q8, 'Q8: frozen pre-trained ResNet-18 + trained FC head augmented') # train/val/test curves

# Comparison with best MLP (Q5) and CNN (Q7)
comp = pd.concat([summarize([('CNN (Q7, aug)', stats_cnn_aug)]), summarize(runs_q8)], ignore_index=True) # CNN row + pre-trained row
mlp_s = runs_q5[-1][1] # stats dict of the augmented MLP from Q5
b = int(np.argmax(mlp_s['val_acc'])) # epoch with best validation accuracy
comp.loc[len(comp)] = ['Best MLP (Q5, aug)', b + 1, mlp_s['val_acc'][b], MLP_TEST_ACC,
                       mlp_s['train_acc'][-1], mlp_s['elapsed'][-1],
                       mlp_s['elapsed'][-1] / len(mlp_s['elapsed'])]
print(comp) # accuracy and training-time comparison
plot_runs([('CNN (Q7, aug)', stats_cnn_aug), ('Pre-trained (Q8)', stats_pre)], 'Q7 vs Q8') #sidebyside

## Q9

In [ ]:
# CIFAR-10 split (10% val)
cifar_raw = datasets.CIFAR10('./data', train=True, download=True) # raw CIFAR-10 training images, just for the statistics
_perm = np.random.default_rng(SEED).permutation(len(cifar_raw))
_tr = _perm[int(VAL_SPLIT * len(cifar_raw)):]
_px = cifar_raw.data[_tr].astype(np.float32) / 255.0 # training pixels in [0,1], shape (N, 32, 32, 3)
C_MEAN, C_STD = _px.mean(axis=(0, 1, 2)).tolist(), _px.std(axis=(0, 1, 2)).tolist() # per colour  channel mean and std

# resize to 64, ImageNet stats
tf_eval_c_pre = transforms.Compose([transforms.Resize(RES), transforms.ToTensor(), norm_imn])
tf_aug_c_pre = transforms.Compose(build_augmentations() + [transforms.Resize(RES), transforms.ToTensor(), norm_imn])
c_train, c_val, c_test, c_train_aug, _, _ = build_sets(datasets.CIFAR10, tf_eval_c_pre, tf_aug_c_pre)
CF_train, CF_val, CF_test = (extract_features(backbone, d) for d in (c_train, c_val, c_test))
cfeat_dls = {'train': loader(CF_train, bs=1000), 'val': loader(CF_val, bs=1000), 'test': loader(CF_test, bs=1000)}

torch.manual_seed(SEED)
head_c = make_head(BEST_DEPTH) # same head design as chosen for FashionMNIST
stats_pre_cifar = train_torch(Pretrained(backbone, head_c), loader(c_train_aug, shuffle=True, workers=WORKERS), cfeat_dls, # train on augmented CIFAR
                              EPOCHS_PRE, lr=HEAD_LR, eval_model=head_c)

# CNN on CIFAR-10 (3 input channels, 32x32)
norm_c = transforms.Normalize(C_MEAN, C_STD)
tf_eval_c = transforms.Compose([transforms.ToTensor(), norm_c])
tf_aug_c = transforms.Compose(build_augmentations() + [transforms.ToTensor(), norm_c])
cs_train, cs_val, cs_test, cs_train_aug, _, _ = build_sets(datasets.CIFAR10, tf_eval_c, tf_aug_c)
cs_eval_dls = {'train': loader(cache_tensors(cs_train), bs=1000), 'val': loader(cache_tensors(cs_val), bs=1000),
               'test': loader(cache_tensors(cs_test), bs=1000)}
torch.manual_seed(SEED)
stats_cnn_cifar = train_torch(CNN(in_ch=3, img=32, c1=BEST_C1, c2=BEST_C2, k=BEST_K), # CNN adapted to 3x32x32 input
                              loader(cs_train_aug, shuffle=True, workers=WORKERS), cs_eval_dls, EPOCHS_CNN, lr=BEST_LR)

In [ ]:
runs_q9 = [('FashionMNIST: pre-trained', stats_pre), ('CIFAR-10: pre-trained', stats_pre_cifar),
           ('FashionMNIST: CNN from scratch', stats_cnn_aug), ('CIFAR-10: CNN from scratch', stats_cnn_cifar)]
plot_runs(runs_q9, 'Q9: pre-trained vs. scratch CNN on two datasets (augmented training)') # curves for all four
tab9 = summarize(runs_q9); print(tab9) # results table

# test acc bar chart
fig, ax = plt.subplots(figsize=(6, 4))
w = 0.35
ax.bar(np.arange(2) - w / 2, [tab9['test acc'][2], tab9['test acc'][3]], w, label='CNN from scratch') # CNN FashionMNIST, CIFAR-10
ax.bar(np.arange(2) + w / 2, [tab9['test acc'][0], tab9['test acc'][1]], w, label='Frozen ResNet-18 + FC') # pre trained FashionMNIST, CIFAR-10
ax.set_xticks(range(2)); ax.set_xticklabels(['FashionMNIST', 'CIFAR-10']); ax.set_ylabel('Test accuracy'); ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()


## Q10

In [ ]:
plot_runs([('CNN (Q6)', stats_cnn), ('CNN + aug (Q7)', stats_cnn_aug), ('Pre-trained + aug (Q8)', stats_pre)],
          'Q6-Q8 overview: where does each model start to overfit?')
print(summarize([('CNN (Q6)', stats_cnn), ('CNN + aug (Q7)', stats_cnn_aug), ('Pre-trained + aug (Q8)', stats_pre)]))

## NxN Confusion Matrix for MLP



In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm_mlp = MLP(hidden=(256, 256), a_func='relu', seed=SEED)
cm_opt = GradientDescent(lr=0.1, batch_size=128, nb_epochs=50)
cm_mlp.fit(X_train, y_train, 10, cm_opt, logging=False)

# generate predictions on the test set
y_pred = cm_mlp.predict(X_test)

# FashionMNIST class labels (indices 0 to 9)
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

# compute and plot the 10x10 confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap="Blues", ax=ax, xticks_rotation=45, colorbar=False)
ax.set_title("FashionMNIST Test Set Confusion Matrix (MLP)")
plt.tight_layout()
plt.show()